In [1]:
%pip install transformers datasets peft scikit-learn accelerate

import numpy as np
from datasets import Dataset
from sklearn.metrics import accuracy_score

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
)

# =========================
# 1. Dummy dataset
# =========================

texts = [
    "I hated this movie",
    "The product was terrible",
    "Very bad experience",

    "The package arrived today",
    "It works as expected",
    "The meeting was normal",

    "I loved this movie",
    "Amazing product",
    "Excellent experience",

    "The service was awful",
    "I am disappointed",
    "Worst result ever",

    "This is okay",
    "Nothing special happened",
    "Average quality",

    "Fantastic performance",
    "Really happy with it",
    "Best purchase ever",
]

labels = [
    0, 0, 0,  # negative
    1, 1, 1,  # neutral
    2, 2, 2,  # positive

    0, 0, 0,
    1, 1, 1,
    2, 2, 2,
]

# =========================
# 2. Tokenizer
# =========================

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

encodings = tokenizer(
    texts,
    truncation=True,
    padding="max_length",
    max_length=64,
)

dataset = Dataset.from_dict({
    "input_ids": encodings["input_ids"],
    "attention_mask": encodings["attention_mask"],
    "labels": labels,
})

# =========================
# 3. Train / validation / test split
# =========================

train_test = dataset.train_test_split(test_size=0.3, seed=42)
valid_test = train_test["test"].train_test_split(test_size=0.5, seed=42)

train_data = train_test["train"]
val_data = valid_test["train"]
test_data = valid_test["test"]

# =========================
# 4. Load BERT model
# =========================

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3,
)

# =========================
# 5. Apply LoRA with PEFT
# =========================

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"],
)

model = get_peft_model(model, lora_config)

# Shows which parameters will be trained
model.print_trainable_parameters()

# =========================
# 6. Metrics
# =========================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy}

# =========================
# 7. Training arguments
# =========================

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    report_to="none",
)

# =========================
# 8. Trainer
# =========================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    compute_metrics=compute_metrics,
)

# =========================
# 9. Train
# =========================

trainer.train()

# =========================
# 10. Evaluate
# =========================

results = trainer.evaluate(eval_dataset=test_data)

print(f"Test Accuracy: {results['eval_accuracy']:.4f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 9.1 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3474.49it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSI

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.112279,0.333333
2,No log,1.118650,0.333333
3,No log,1.121796,0.333333


/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
No log,1.125504,3,0.333333


Test Accuracy: 0.3333
